<a href="https://colab.research.google.com/github/AchalaJain/telco-llm-analytics-assistant/blob/main/notebooks/01_llm_text_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-auth==2.49.0

#Setting up the data + LLM API access

In [2]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

print("API key loaded successfully:", api_key is not None)

API key loaded successfully: True


In [3]:
from google import genai

client = genai.Client(api_key=api_key)

In [27]:
# response = client.models.generate_content(
#     model="gemini-3.6-flash",
#     contents="Explain SQL in one simple sentence."
# )

# print(response.text)

In [7]:
import pandas as pd

df = pd.read_csv("/content/telco_churn_cleaned.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_Flag,Revenue_at_risk
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0.00
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,0,0.00
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,53.85
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,0.00
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,70.70


In [8]:
df.columns.to_list()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'Churn_Flag',
 'Revenue_at_risk']

In [9]:
df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [10]:
df.shape

(7032, 23)

In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("telco.db")

df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully.")

Database created successfully.


In [12]:
result = pd.read_sql_query(
    "SELECT * FROM customers LIMIT 5",
    conn
)

display(result)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_Flag,Revenue_at_risk
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0.00
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,0,0.00
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,53.85
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,0.00
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,70.70


In [13]:
count = pd.read_sql_query(
    "SELECT COUNT(*) AS customer_count FROM customers",
    conn
)

display(count)

,customer_count
0,7032


In [14]:
schema_df = pd.read_sql_query(
    "PRAGMA table_info(customers)",
    conn
)

display(schema_df)

,cid,name,type,notnull,dflt_value,pk
0,0,customerID,TEXT,0,None,0
1,1,gender,TEXT,0,None,0
2,2,SeniorCitizen,INTEGER,0,None,0
3,3,Partner,TEXT,0,None,0
4,4,Dependents,TEXT,0,None,0
5,5,tenure,INTEGER,0,None,0
6,6,PhoneService,TEXT,0,None,0
7,7,MultipleLines,TEXT,0,None,0
8,8,InternetService,TEXT,0,None,0
9,9,OnlineSecurity,TEXT,0,None,0


#Schema-aware prompt

In [15]:
SCHEMA = """
Table: customers

Customer information:
- customerID (TEXT) -- unique customer identifier
- gender (TEXT) -- customer gender
- SeniorCitizen (INTEGER) -- 1 if senior citizen, 0 otherwise
- Partner (TEXT) -- whether customer has a partner: 'Yes' or 'No'
- Dependents (TEXT) -- whether customer has dependents: 'Yes' or 'No'

Customer tenure and services:
- tenure (INTEGER) -- number of months the customer has been with the company
- PhoneService (TEXT) -- whether customer has phone service
- MultipleLines (TEXT) -- multiple phone lines status
- InternetService (TEXT) -- type of internet service
- OnlineSecurity (TEXT) -- online security service status
- OnlineBackup (TEXT) -- online backup service status
- DeviceProtection (TEXT) -- device protection service status
- TechSupport (TEXT) -- technical support service status
- StreamingTV (TEXT) -- streaming TV service status
- StreamingMovies (TEXT) -- streaming movies service status

Contract and billing:
- Contract (TEXT) -- contract type: 'Month-to-month', 'One year', 'Two year'
- PaperlessBilling (TEXT) -- whether paperless billing is enabled
- PaymentMethod (TEXT) -- customer's payment method
- MonthlyCharges (REAL) -- customer's monthly charge
- TotalCharges (REAL) -- total charges over the customer's tenure

Churn:
- Churn (TEXT) -- whether the customer churned: 'Yes' or 'No'
- Churn_Flag (INTEGER) -- binary churn indicator: 1 if churned, 0 otherwise
- Revenue_at_risk (REAL) -- monthly revenue at risk; equals MonthlyCharges for churned customers and 0 for customers who have not churned
"""

In [16]:
SYSTEM_PROMPT = f"""
You are an expert data analyst and SQL generator.

Your task is to convert the user's natural-language
analytics question into a single valid SQLite SELECT query.

DATABASE SCHEMA:

{SCHEMA}

RULES:

1. Generate exactly one SQL SELECT statement.

2. Only query the customers table.

3. Never generate INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, REPLACE, TRUNCATE, or other data-modifying statements.

4. Only use tables and columns explicitly provided in the schema.

5. Never invent columns, tables, metrics, or values.

6. Use SQLite-compatible SQL syntax.

7. Use the exact categorical values provided in the schema.

8. For churn analysis, you may use either:
   - Churn = 'Yes' / 'No'
   - Churn_Flag = 1 / 0
   depending on which is more appropriate.

9. When calculating churn rate, calculate:
   churned customers / total customers.

10. When calculating percentages or rates, use floating-point
    arithmetic to avoid integer division.

11. When ranking groups, order the result appropriately.

12. If the user asks for "highest", "lowest", "top", or "bottom",
    make the ordering explicit.

13. If the user's question is ambiguous because an important
    metric, dimension, filter, or ranking criterion is unspecified,
    do not invent a business definition. Use the most defensible
    interpretation only when the intent is reasonably clear.

14. Do not assume columns or business metrics that are not present
    in the schema.

15. Return ONLY the SQL query.

16. Do not return explanations.

17. Do not use markdown formatting.

18. Do not wrap the SQL in ```sql or ``` code fences.

IMPORTANT BUSINESS DEFINITIONS:

- Churn_Flag = 1 means the customer churned.
- Churn_Flag = 0 means the customer did not churn.
- Revenue_at_risk represents monthly revenue at risk due to churn.
- Revenue_at_risk = MonthlyCharges for churned customers and 0 otherwise.
- Therefore, SUM(Revenue_at_risk) represents total monthly revenue currently at risk from churn.
"""

In [17]:
def question_to_sql(user_question: str) -> str:

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0,
        ),
        contents=user_question,
    )

    return response.text.strip()

In [18]:
from google.genai import types

In [19]:
sql = question_to_sql(
    "What is the average monthly charge for customers who churned?"
)

print(sql)

SELECT AVG(MonthlyCharges) FROM customers WHERE Churn = 'Yes'


In [20]:
result = pd.read_sql_query(sql, conn)

display(result)

,AVG(MonthlyCharges)
0,74.441332


In [19]:
def test_question(question: str):

    print("=" * 80)
    print("QUESTION")
    print(question)

    sql = question_to_sql(question)

    print("\nGENERATED SQL")
    print(sql)

    try:
        result = pd.read_sql_query(sql, conn)

        print("\nQUERY RESULT")
        display(result)

    except Exception as e:

        print("\nSQL EXECUTION ERROR")
        print(e)

In [22]:
#Simple count
test_question(
    "How many customers are on a two-year contract?"
)

QUESTION
How many customers are on a two-year contract?

GENERATED SQL
SELECT COUNT(*) FROM customers WHERE Contract = 'Two year'

QUERY RESULT


,COUNT(*)
0,1685


In [ ]:
#Average
test_question(
    "What is the average monthly charge?"
)

In [ ]:
#Churn by contract
test_question(
    "Which contract type has the most churned customers?"
)

In [ ]:
#Churn rate
test_question(
    "Which contract type has the highest churn rate?"
)

In [32]:
#Multi-dimensional
test_question(
    "Which combination of contract type and internet service has the highest churn rate?"
)

QUESTION
Which combination of contract type and internet service has the highest churn rate?

GENERATED SQL
SELECT Contract, InternetService, 1.0 * SUM(Churn_Flag) / COUNT(*) AS churn_rate FROM customers GROUP BY Contract, InternetService ORDER BY churn_rate DESC LIMIT 1

QUERY RESULT


,Contract,InternetService,churn_rate
0,Month-to-month,Fiber optic,0.546053


In [34]:
#Deliberately ambigous
test_question(
    "Show me the top customers."
)

QUESTION
Show me the top customers.

GENERATED SQL
SELECT * FROM customers ORDER BY TotalCharges DESC LIMIT 10

QUERY RESULT


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_Flag,Revenue_at_risk
0,2889-FPWRM,Male,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,One year,Yes,Bank transfer (automatic),117.80,8684.80,Yes,1,117.8
1,7569-NMZYQ,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Bank transfer (automatic),118.75,8672.45,No,0,0.0
2,9739-JLPQJ,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,No,Credit card (automatic),117.50,8670.10,No,0,0.0
3,9788-HNGUT,Male,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),116.95,8594.40,No,0,0.0
4,8879-XUAHX,Male,0,Yes,No,71,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Electronic check,116.25,8564.75,No,0,0.0
5,9924-JPRMC,Male,0,No,No,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Electronic check,118.20,8547.15,No,0,0.0
6,0675-NCDYU,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),116.40,8543.25,No,0,0.0
7,6650-BWFRT,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Credit card (automatic),117.15,8529.50,No,0,0.0
8,0164-APGRB,Female,0,No,No,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Electronic check,114.90,8496.70,No,0,0.0
9,1488-PBLJN,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Two year,Yes,Bank transfer (automatic),116.85,8477.70,No,0,0.0


In [35]:

#Monthly Revenue at risk
test_question(
    "How much monthly revenue is currently at risk due to churn?"
)

QUESTION
How much monthly revenue is currently at risk due to churn?

GENERATED SQL
SELECT SUM(Revenue_at_risk) FROM customers

QUERY RESULT


,SUM(Revenue_at_risk)
0,139130.85


In [41]:
#Revenue at risk
test_question(
    "Which contract type has the highest revenue at risk?"
)

QUESTION
Which contract type has the highest revenue at risk?


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [38]:
#Average revenue at risk per customer
test_question(
    "Which contract type has the highest average revenue at risk per customer?"
)

QUESTION
Which contract type has the highest average revenue at risk per customer?


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
#Churn rate for each contract type
test_question(
    "What is the churn rate for each contract type?"
)

In [ ]:
#Ambigous on customer segment
test_question(
    "Which customer segment has the highest churn risk?"
)

# SQL Validation & Safe Execution

In [39]:
import re

FORBIDDEN_KEYWORDS = [
    "INSERT",
    "UPDATE",
    "DELETE",
    "DROP",
    "ALTER",
    "TRUNCATE",
    "ATTACH",
    "PRAGMA"
]


def validate_sql(sql: str) -> bool:

    sql_upper = sql.upper().strip()

    # Query must start with SELECT
    if not sql_upper.startswith("SELECT"):
        return False

    # Check for forbidden SQL operations
    for keyword in FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{keyword}\b", sql_upper):
            return False

    return True

In [40]:
validate_sql(
    "SELECT COUNT(*) FROM customers"
)

True